# Elasticity equation - compute CFF

In this tutorial we present how to solve the elasticity equation with [PorePy](https://github.com/pmgbergen/porepy) and then how to post process the tractions to compute the CFF (Coulomb friction failure) coefficient along a fracture. The unknown is the displacement $u$.

Let $\Omega=(0,1)^3$ with boundary $\partial \Omega$ and outward unit normal ${\nu}$. Given 
$\lambda$ Lamé constant and $\mu$ the Kirchhoff modulus, we want to solve the following problem: find $u$ such that
$$
\nabla \cdot [ 2 \mu \epsilon(u) + \lambda \nabla \cdot u] = -b
$$
with $\epsilon$ the symmetric gradient and $b$ a body force. The CFF is computed as
$$
    CFF = \tau^\top \sigma n + \mu_f n^\top \sigma n
$$
with $\mu_f$ the fracture friction coefficient. We have the following possibilities
- No slip: $CFF < 0$
- Slipping: $CFF \geq 0$

We will use the Multi-Point Stress Approximation (MPSA) to discretise the problem.

## Exercise 2: dam filling

The dam filling problem is when a (time dependent) force is impose on the top compressing the body, the bottom is fixed, and lateral compressive forces are imposed on the other sides of the boundary.

For this test case we set $\Omega = [0, 1]^3$, $b = 0$, and the following boundary conditions:
$$ 
u = 0 \text{ on } \partial_{bottom} \Omega \qquad \nu \cdot \sigma = [f_{lrfb}, 0] \text{ on } \partial_{left} \Omega \cup \partial_{right} \Omega \cup \partial_{front}\Omega \cup \partial_{back} \Omega \qquad \nu \cdot \sigma = [0, f_t]^\top \text{ on } \partial_{top} \Omega
$$

whith $f_{lrfb} < 0 $ as well as $f_t < 0$.

We have the following relations between the $\lambda$ and $\mu$ as functions of the Young's modulus $E$ and Poisson ratio $\nu$
$$
\lambda = \dfrac{\nu E}{(1+\nu)(1-2\nu)}\qquad \mu = \dfrac{E}{2(1+\nu)}
$$

This is the guided ("fill in the code") version of `ex2.ipynb` -- work through the cells in order, completing each `__TODO__`. Compare against `ex2.ipynb` once you're done, or if you get stuck.

First we import some of the standard modules, like `numpy` and `scipy.sparse`. Since PyGeoN is based on [PorePy](https://github.com/pmgbergen/porepy) we import both modules.

In [ ]:
import numpy as np
import scipy.sparse as sps

import porepy as pp
import pygeon as pg

We create now the grid and the fracture/fault now represent an internal constraint so that the grid adapts to it.

In [ ]:
N = 10
dim = 3
mesh_size = 1 / N

# Define a fracture
frac_pts = np.array(
    [
        [0.2, 0.9, 0.9, 0.2],
        [0.2, 0.2, 0.8, 0.8],
        [0.2, 1, 1, 0.2],
    ]
)
frac = pp.PlaneFracture(frac_pts)

sd = pg.unit_grid(
    dim, mesh_size, as_mdg=False, fractures=[frac], constraints=np.array([0])
)
sd.compute_geometry()

With the following code we set the data, in particular the Lamé and the Kirchhoff modulus, and the boundary conditions. Since we need to identify each side of $\partial \Omega$ we need few steps.

In [ ]:
key = "elasticity"

E = 1
nu = 0.25
mu_fric = 0.45

# forces imposed at the boundary
fun_top = -1e-2  # -1e-2 (no slip) or -2.5e-2 (some slip) or -3e-2 (slip)
fun_left = 1e-2
fun_right = -1e-2
fun_front = -1e-2
fun_back = 1e-2

# TODO: compute lambda_ and mu from E and nu using the relations above
lambda_ = __TODO__
mu = __TODO__

# Create stiffness matrix
lambda_ = lambda_ * np.ones(sd.num_cells)
mu = mu * np.ones(sd.num_cells) / 2
C = pp.FourthOrderTensor(mu, lambda_)

# Define boundary type
b_faces = sd.get_all_boundary_faces()
num_b_faces = b_faces.size
labels = np.array(["neu"] * num_b_faces)

bottom = np.isclose(sd.face_centers[2, b_faces], 0)
labels[bottom] = "dir"
bound = pp.BoundaryConditionVectorial(sd, b_faces, labels)

bc_values = np.zeros((sd.dim, sd.num_faces))

# TODO: impose the compressive/top forces (fun_top/left/right/front/back,
# each scaled by the corresponding face area) on the six sides identified
# via sd.face_centers, following the same pattern used elsewhere in the
# course for identifying boundary faces


bc_values = bc_values.ravel("F")

# No source term
source = np.zeros(sd.num_cells * sd.dim)

# collect all data
param = {
    "fourth_order_tensor": C,
    "bc_values": bc_values,
    "bc": bound,
    "source": source,
}
data = pp.initialize_data({}, key, param)

Once the data are assigned to the grid, we construct the matrices. Once the latter is created, we also construct the right-hand side containing the boundary conditions.

In [ ]:
# TODO: discretize with MPSA and solve the linear system for the displacement u
mpsa = pp.Mpsa(key)
mpsa.discretize(sd, data)

A, b = __TODO__
u = __TODO__

We first compute the traction for each face of the cell and reshape it so that it has the x-components for all the faces first and then all the y-components (and then all z-components in 3d case).

In [ ]:
# post process the traction for each face
mat = data[pp.DISCRETIZATION_MATRICES][key]
mat_stress = mat[mpsa.stress_matrix_key]
mat_bound_stress = mat[mpsa.bound_stress_matrix_key]

# TODO: compute the traction t (the measure is in Pascals)
t = __TODO__

# reshape the traction to be in the order of first all the x-components, then all the
# y-components, and finally the z-components
t = np.reshape(t, (sd.dim, -1), order="F").ravel()

We restrict the traction to the fracture faces

In [ ]:
# compute the faces that are on the fracture
points = sd.face_centers
dist, _, _ = pp.distances.points_polygon(points, frac_pts)
faces_on_frac = np.isclose(dist, 0)

# TODO: extract the components of the traction on the fracture faces and
# reshape them into a (sd.dim, -1) array
t_on_frac = __TODO__

We compute now the normal and tangential projection matrices, so that given a vector it aligns it accordingly.

In [ ]:
# compute the unit normal vector to the fracture
normal = sd.face_normals[:, faces_on_frac]
normal /= np.linalg.norm(normal, axis=0)

# TODO: compute the normal projection matrix (outer product of normal with
# itself, per face) and the tangential projection matrix (identity minus
# the normal projection)
normal_proj = __TODO__
tangential_proj = __TODO__

Let us compute now the normal and tangential parts of the traction and evaluate the CFF.

In [ ]:
# TODO: compute the normal component of the traction (project t_on_frac with
# normal_proj, then dot with normal)
t_normal_vec = __TODO__
t_normal = __TODO__

# TODO: compute the tangential component of the traction (project t_on_frac
# with tangential_proj, then take its norm)
t_tangential_vec = __TODO__
t_tangential = __TODO__

We finally export the solution to be visualized by [ParaView](https://www.paraview.org/).

In [ ]:
# TODO: compute the Coulomb friction factor CFF = t_tangential + mu_fric * t_normal
cff = __TODO__

# determine which face slips
faces_slip = cff >= 0

print("Number of slipping faces:", np.sum(faces_slip), "over", cff.size)

Finally, we compute the CFF and report it

In [ ]:
# reshape the displacement for the export
u = np.reshape(u, (sd.dim, -1), order="F")

save = pp.Exporter(sd, "sol", folder_name="ex2")
save.write_vtu([("cell_u", u)])

We verify that the computed CFF matches the expected reference value.

In [ ]:
# Consistency check -- once your implementation is correct, this should pass
assert np.isclose(cff.sum(), -0.00260037801235645)